<!-- NOTEBOOK_OVERVIEW -->
# 1. Transformer Encoder Baselines (RoBERTa + DeBERTa)

## 2. Introduction
This notebook adds supervised transformer classifier baselines to the dissertation comparison stack. It fine-tunes two modern encoder models on the in-distribution training split, selects the deployment threshold on `VAL` only, and evaluates them on `TEST` plus all fixed OOD sets under the same no-OOD-tuning protocol used elsewhere in the repository.

## 3. Workflow Steps
1. Load the active processed split artifact (`A`, `B`, or `C`) and construct the shared ID/OOD views.
2. Fine-tune `roberta-base` and `microsoft/deberta-base` as supervised text classifiers using only `train` and `val`.
3. Choose threshold `t*` on `VAL` by maximizing macro-F1 and freeze it for `TEST` and OOD evaluation.
4. Export canonical split-level metrics and optional per-OOD/bin diagnostics using the same result schema as the existing baselines.

## 4. Evaluation and Protocol Notes
1. OOD sets are evaluation-only and are never used for training, threshold tuning, or model selection.
2. Reported metrics include `Accuracy`, `Macro-F1`, `ROC-AUC`, `AUC-PR`, and `TPR@{1%,5%,10%} FPR`.
3. Difficulty-bin outputs summarize behavior by prompt length and lexical complexity quartiles.
4. These models are treated as modern supervised encoder baselines, not as routed systems, so `semantic_coverage` and `defer_rate` remain empty.

## 5. Execution Notes
1. Set `SPLIT_TAG` to `A`, `B`, or `C` before execution.
2. Use `WRITE_MINIMAL_OUTPUTS=0` to regenerate secondary OOD and bin-level diagnostics.
3. Model downloads come from Hugging Face the first time each backbone is used in the local environment.


In [1]:
# Cell Purpose: Import required libraries, project modules, and shared utilities.
# 1) Imports + config

from pathlib import Path
import gc
import os
import sys

import numpy as np
import pandas as pd
import torch

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)


In [2]:
# Cell Purpose: Load processed dataset partitions and experiment helpers.
# 2) Dataset + evaluation setup

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.evaluation.eval_metrics import (
    evaluate_predictions,
    results_to_dataframe,
    best_threshold_by_macro_f1,
)
from src.common.notebook_utils import safe_qcut, text_stats
from src.models import TransformerBaselineConfig, train_transformer_baseline, predict_transformer_probabilities

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"

SPLIT_TAG = os.getenv("SPLIT_TAG", "B").strip().upper()  # env override: A/B/C
WRITE_MINIMAL_OUTPUTS = os.getenv("WRITE_MINIMAL_OUTPUTS", "1").strip().lower() not in {"0", "false", "no"}
print("WRITE_MINIMAL_OUTPUTS:", WRITE_MINIMAL_OUTPUTS)

expected_filename_by_split = {
    "A": "jailbreak_benchmarks_processed_v2.csv",
    "B": "jailbreak_benchmarks_processed_v2_splitB.csv",
    "C": "jailbreak_benchmarks_processed_v2_splitC.csv",
}
if SPLIT_TAG not in expected_filename_by_split:
    raise ValueError("SPLIT_TAG must be 'A', 'B', or 'C'.")
processed_filename = expected_filename_by_split[SPLIT_TAG]

processed_path = DATA_PROCESSED / processed_filename
print("Using dataset:", processed_path)

df = pd.read_csv(processed_path)

print("Rows:", len(df))
print()
print("Split counts:")
print(df["split"].value_counts())
print()
print("Label counts:")
print(df["label"].value_counts())


df_train = df[df["split"] == "train"].copy()
df_val = df[df["split"] == "val"].copy()
df_test = df[df["split"] == "test"].copy()

OOD_SPLIT_ORDER = ["ood_test", "ood_test_injection", "ood_test_injection_standard"]
df_ood_map = {}
for split_name in OOD_SPLIT_ORDER:
    d = df[df["split"] == split_name].copy()
    if not d.empty:
        df_ood_map[split_name] = d

if "ood_test" not in df_ood_map:
    raise ValueError("Missing required split 'ood_test'.")

for name, d in [("train", df_train), ("val", df_val), ("test", df_test)] + list(df_ood_map.items()):
    print(f"{name:24s}", d.shape, d["label"].value_counts().to_dict())

X_train_text = df_train["prompt_text"].astype(str).tolist()
X_val_text = df_val["prompt_text"].astype(str).tolist()
X_test_text = df_test["prompt_text"].astype(str).tolist()
X_ood_text_map = {name: d["prompt_text"].astype(str).tolist() for name, d in df_ood_map.items()}

y_train = df_train["label"].to_numpy(dtype=int)
y_val = df_val["label"].to_numpy(dtype=int)
y_test = df_test["label"].to_numpy(dtype=int)
y_ood_map = {name: d["label"].to_numpy(dtype=int) for name, d in df_ood_map.items()}


WRITE_MINIMAL_OUTPUTS: False
Using dataset: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/data/processed/jailbreak_benchmarks_processed_v2_splitC.csv
Rows: 6424

Split counts:
split
ood_test_injection_standard    3986
ood_test                        768
train                           694
ood_test_injection              678
test                            149
val                             149
Name: count, dtype: int64

Label counts:
label
1    3437
0    2987
Name: count, dtype: int64
train                    (694, 16) {1: 504, 0: 190}
val                      (149, 16) {1: 109, 0: 40}
test                     (149, 16) {1: 108, 0: 41}
ood_test                 (768, 16) {0: 384, 1: 384}
ood_test_injection       (678, 16) {0: 339, 1: 339}
ood_test_injection_standard (3986, 16) {1: 1993, 0: 1993}


In [3]:
# Cell Purpose: Fine-tune transformer baselines and evaluate all required splits.
# 3) Train and score RoBERTa / DeBERTa baselines

MODEL_CONFIGS = [
    TransformerBaselineConfig(
        model_name="roberta-base",
        output_label="TRANSFORMER_ROBERTA_BASE",
        max_length=256,
        learning_rate=2e-5,
        num_epochs=3,
        train_batch_size=8,
        eval_batch_size=16,
        weight_decay=0.01,
        warmup_ratio=0.1,
    ),
    TransformerBaselineConfig(
        model_name="microsoft/deberta-base",
        output_label="TRANSFORMER_DEBERTA_BASE",
        max_length=256,
        learning_rate=2e-5,
        num_epochs=3,
        train_batch_size=8,
        eval_batch_size=16,
        weight_decay=0.01,
        warmup_ratio=0.1,
    ),
]

all_metric_frames = []
secondary_ood_frames = {}
transformer_history_frames = []
inference_cache = {}

for model_cfg in MODEL_CONFIGS:
    print()
    print(f"=== Training {model_cfg.output_label} ({model_cfg.model_name}) ===")
    artifacts = train_transformer_baseline(
        X_train_text,
        y_train,
        X_val_text,
        y_val,
        config=model_cfg,
        seed=RANDOM_SEED,
    )

    val_proba = artifacts.val_probabilities
    t_star, best_val_f1 = best_threshold_by_macro_f1(y_val, val_proba, n_grid=1001)
    val_pred_at_t = (val_proba >= t_star).astype(int)
    val_neg = (y_val == 0)
    val_fpr_at_t = float(((val_pred_at_t == 1) & val_neg).sum() / max(val_neg.sum(), 1))

    print(f"Selected threshold on VAL for {model_cfg.output_label}: t*={t_star:.3f}")
    print(f"Best VAL macro-F1 at t*: {best_val_f1:.4f}")
    print(f"VAL FPR at t*: {val_fpr_at_t:.4f}")

    test_proba = predict_transformer_probabilities(
        artifacts.model,
        artifacts.tokenizer,
        X_test_text,
        batch_size=model_cfg.eval_batch_size,
        max_length=model_cfg.max_length,
        device=artifacts.device,
    )
    ood_proba_map = {
        ood_name: predict_transformer_probabilities(
            artifacts.model,
            artifacts.tokenizer,
            texts,
            batch_size=model_cfg.eval_batch_size,
            max_length=model_cfg.max_length,
            device=artifacts.device,
        )
        for ood_name, texts in X_ood_text_map.items()
    }

    history_df = pd.DataFrame(artifacts.history)
    history_df.insert(0, "model", model_cfg.output_label)
    history_df.insert(1, "backbone_name", model_cfg.model_name)
    history_df.insert(2, "split_tag", SPLIT_TAG)
    transformer_history_frames.append(history_df)

    note_base = (
        f"policy=macro_f1; t*={t_star:.3f}; val_fpr={val_fpr_at_t:.4f}; "
        f"score=transformer_proba; backbone={model_cfg.model_name}; max_len={model_cfg.max_length}; "
        f"epochs={model_cfg.num_epochs}; lr={model_cfg.learning_rate:.0e}; device={artifacts.device}"
    )

    def eval_from_scores(split_name: str, y_true: np.ndarray, y_score: np.ndarray, *, ood_name: str) -> pd.DataFrame:
        y_pred = (y_score >= t_star).astype(int)
        note = note_base if ood_name == "id" else f"{note_base}; ood_name={ood_name}"
        result = evaluate_predictions(
            split_name=split_name,
            y_true=y_true,
            y_pred=y_pred,
            y_score_for_metrics=y_score,
            print_report=True,
            threshold_note=note,
        )
        df_out = results_to_dataframe(model_cfg.output_label, [result])
        df_out["eval_track"] = "deployment_threshold"
        df_out["ood_name"] = ood_name
        df_out["backbone_name"] = model_cfg.model_name
        df_out["max_length"] = int(model_cfg.max_length)
        df_out["learning_rate"] = float(model_cfg.learning_rate)
        df_out["num_epochs"] = int(model_cfg.num_epochs)
        df_out["train_batch_size"] = int(model_cfg.train_batch_size)
        df_out["eval_batch_size"] = int(model_cfg.eval_batch_size)
        df_out["weight_decay"] = float(model_cfg.weight_decay)
        df_out["warmup_ratio"] = float(model_cfg.warmup_ratio)
        df_out["training_seed"] = int(RANDOM_SEED)
        df_out["device"] = artifacts.device
        return df_out

    val_df = eval_from_scores("VAL", y_val, val_proba, ood_name="id")
    test_df = eval_from_scores("TEST", y_test, test_proba, ood_name="id")
    ood_primary_df = eval_from_scores("OOD", y_ood_map["ood_test"], ood_proba_map["ood_test"], ood_name="ood_test")

    all_metric_frames.extend([val_df, test_df, ood_primary_df])

    for ood_name, y_ood_curr in y_ood_map.items():
        if ood_name == "ood_test":
            continue
        ood_extra_df = eval_from_scores("OOD", y_ood_curr, ood_proba_map[ood_name], ood_name=ood_name)
        secondary_ood_frames.setdefault(ood_name, []).append(ood_extra_df)

    inference_cache[model_cfg.output_label] = {
        "threshold": float(t_star),
        "note_base": note_base,
        "config": model_cfg,
        "val_score": val_proba,
        "test_score": test_proba,
        "ood_score_map": ood_proba_map,
    }

    del artifacts
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

results_df = pd.concat(all_metric_frames, ignore_index=True)
secondary_ood_results = {
    ood_name: pd.concat(frames, ignore_index=True)
    for ood_name, frames in secondary_ood_frames.items()
}
transformer_history_df = pd.concat(transformer_history_frames, ignore_index=True)

print()
print("=== Transformer Baseline Summary (primary OOD) ===")
display(results_df)
if secondary_ood_results:
    print()
    print("=== Transformer Baseline Secondary OOD Rows ===")
    display(pd.concat(list(secondary_ood_results.values()), ignore_index=True))
print()
print("=== Training Curves ===")
display(transformer_history_df)



=== Training TRANSFORMER_ROBERTA_BASE (roberta-base) ===


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Selected threshold on VAL for TRANSFORMER_ROBERTA_BASE: t*=0.396
Best VAL macro-F1 at t*: 0.9742
VAL FPR at t*: 0.0500



=== VAL ===
              precision    recall  f1-score   support

           0      0.974     0.950     0.962        40
           1      0.982     0.991     0.986       109

    accuracy                          0.980       149
   macro avg      0.978     0.970     0.974       149
weighted avg      0.980     0.980     0.980       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[ 38   2]
 [  1 108]]
AUC-PR:  0.9992
ROC-AUC: 0.9977
TPR @ FPR: 1%=0.9541, 5%=0.9908, 10%=1.0000

=== TEST ===
              precision    recall  f1-score   support

           0      1.000     0.951     0.975        41
           1      0.982     1.000     0.991       108

    accuracy                          0.987       149
   macro avg      0.991     0.976     0.983       149
weighted avg      0.987     0.987     0.986       149

Confusion matrix [ [TN FP] [FN TP] ]:
 [[ 39   2]
 [  0 108]]
AUC-PR:  0.9951
ROC-AUC: 0.9892
TPR @ FPR: 1%=0.6296, 5%=1.0000, 10%=1.0000

=== OOD ===
              precision    rec

Some weights of DebertaForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-base and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

,model,split,acc,macro_f1,auc_pr,roc_auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,tpr_at_10pct_fpr,semantic_coverage,...,backbone_name,max_length,learning_rate,num_epochs,train_batch_size,eval_batch_size,weight_decay,warmup_ratio,training_seed,device
0,TRANSFORMER_ROBERTA_BASE,VAL,0.979866,0.974163,0.999168,0.997706,0.954128,0.990826,1.000000,None,...,roberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps
1,TRANSFORMER_ROBERTA_BASE,TEST,0.986577,0.982913,0.995105,0.989160,0.629630,1.000000,1.000000,None,...,roberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps
2,TRANSFORMER_ROBERTA_BASE,OOD,0.778646,0.778261,0.901115,0.889825,0.388021,0.598958,0.682292,None,...,roberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps
3,TRANSFORMER_DEBERTA_BASE,VAL,0.986577,0.982634,0.999500,0.998624,0.972477,1.000000,1.000000,None,...,microsoft/deberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps
4,TRANSFORMER_DEBERTA_BASE,TEST,0.973154,0.965826,0.995755,0.990289,0.666667,0.990741,0.990741,None,...,microsoft/deberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps
5,TRANSFORMER_DEBERTA_BASE,OOD,0.798177,0.794767,0.884411,0.880737,0.335938,0.536458,0.601562,None,...,microsoft/deberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps



=== Transformer Baseline Secondary OOD Rows ===


,model,split,acc,macro_f1,auc_pr,roc_auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,tpr_at_10pct_fpr,semantic_coverage,...,backbone_name,max_length,learning_rate,num_epochs,train_batch_size,eval_batch_size,weight_decay,warmup_ratio,training_seed,device
0,TRANSFORMER_ROBERTA_BASE,OOD,0.715339,0.691355,0.855449,0.917713,0.002950,0.454277,0.755162,None,...,roberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps
1,TRANSFORMER_DEBERTA_BASE,OOD,0.702065,0.673042,0.963570,0.979764,0.336283,0.938053,0.994100,None,...,microsoft/deberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps
2,TRANSFORMER_ROBERTA_BASE,OOD,0.661816,0.621006,0.803650,0.835046,0.077270,0.302057,0.448570,None,...,roberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps
3,TRANSFORMER_DEBERTA_BASE,OOD,0.649523,0.602185,0.830422,0.852805,0.119418,0.342198,0.535374,None,...,microsoft/deberta-base,256,0.00002,3,8,16,0.01,0.1,42,mps



=== Training Curves ===


,model,backbone_name,split_tag,epoch,train_loss,val_macro_f1_at_0_5
0,TRANSFORMER_ROBERTA_BASE,roberta-base,C,1.0,0.539241,0.846328
1,TRANSFORMER_ROBERTA_BASE,roberta-base,C,2.0,0.240518,0.947902
2,TRANSFORMER_ROBERTA_BASE,roberta-base,C,3.0,0.102036,0.974163
3,TRANSFORMER_DEBERTA_BASE,microsoft/deberta-base,C,1.0,0.493228,0.894482
4,TRANSFORMER_DEBERTA_BASE,microsoft/deberta-base,C,2.0,0.219183,0.966350
5,TRANSFORMER_DEBERTA_BASE,microsoft/deberta-base,C,3.0,0.098364,0.947013


In [4]:
# Cell Purpose: Persist canonical split-level transformer metrics and optional secondary OOD files.
# 4) Save metrics outputs

OUT_DIR = PROJECT_ROOT / "experiments" / "results" / "metrics"
OUT_DIR.mkdir(parents=True, exist_ok=True)

out_path = OUT_DIR / f"metrics_transformer_split{SPLIT_TAG}.csv"
results_df.to_csv(out_path, index=False)
print(f"Saved primary transformer metrics: {out_path}")

if not WRITE_MINIMAL_OUTPUTS:
    out_primary_suffix = OUT_DIR / f"metrics_transformer_split{SPLIT_TAG}__ood-ood_test.csv"
    results_df.to_csv(out_primary_suffix, index=False)
    print(f"Saved primary OOD-suffixed transformer metrics: {out_primary_suffix}")

    for ood_name, df_extra in secondary_ood_results.items():
        out_extra = OUT_DIR / f"metrics_transformer_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_extra.to_csv(out_extra, index=False)
        print(f"Saved secondary transformer OOD metrics ({ood_name}): {out_extra}")
else:
    print("WRITE_MINIMAL_OUTPUTS=True: skipped transformer OOD-suffixed metrics files.")


Saved primary transformer metrics: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_splitC.csv
Saved primary OOD-suffixed transformer metrics: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_splitC__ood-ood_test.csv
Saved secondary transformer OOD metrics (ood_test_injection): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_splitC__ood-ood_test_injection.csv
Saved secondary transformer OOD metrics (ood_test_injection_standard): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_splitC__ood-ood_test_injection_standard.csv


In [5]:
# Cell Purpose: Produce bin-level diagnostics for transformer baselines.
# 5) Bin-level diagnostics (length + lexical complexity) for TEST/OOD

def _build_bin_rows(
    model_label: str,
    prompts_stage: pd.Series,
    y_stage: np.ndarray,
    y_score_stage: np.ndarray,
    *,
    stage_name: str,
    ood_name: str,
):
    rows = []
    meta = inference_cache[model_label]
    threshold = float(meta["threshold"])
    config = meta["config"]
    note_base = meta["note_base"]
    y_pred_stage = (y_score_stage >= threshold).astype(int)

    stats_stage = text_stats(prompts_stage)
    stats_stage["length_bin"] = safe_qcut(stats_stage["token_count"], q=4, prefix="len")
    stats_stage["complexity_bin"] = safe_qcut(stats_stage["lexical_ttr"], q=4, prefix="complex")

    for bin_family in ["length_bin", "complexity_bin"]:
        for bin_label in sorted([b for b in stats_stage[bin_family].dropna().unique()]):
            mask = (stats_stage[bin_family] == bin_label).to_numpy(dtype=bool)
            y_s = y_stage[mask]
            p_s = y_pred_stage[mask]
            s_s = y_score_stage[mask]

            n_total = int(mask.sum())
            n_pos = int((y_s == 1).sum())
            n_neg = int((y_s == 0).sum())
            if n_total == 0 or n_pos == 0 or n_neg == 0:
                continue

            note = (
                f"{note_base}; bin_eval={bin_family}:{bin_label}; stage={stage_name}; "
                f"eval_track=deployment_threshold; n_total={n_total}; n_pos={n_pos}; n_neg={n_neg}; ood_name={ood_name}"
            )
            res = evaluate_predictions(
                split_name=f"{stage_name}_BIN_{bin_family.upper()}_{str(bin_label).upper()}",
                y_true=y_s,
                y_pred=p_s,
                y_score_for_metrics=s_s,
                print_report=False,
                threshold_note=note,
            )

            evasion_rate = float(((y_s == 1) & (p_s == 0)).sum() / max(n_pos, 1))
            df_one = results_to_dataframe(model_label, [res])
            df_one["eval_track"] = "deployment_threshold"
            df_one["slice_stage"] = stage_name
            df_one["bin_family"] = bin_family
            df_one["bin_label"] = str(bin_label)
            df_one["bin_count"] = n_total
            df_one["bin_positive_count"] = n_pos
            df_one["bin_negative_count"] = n_neg
            df_one["evasion_rate"] = evasion_rate
            df_one["token_count_median"] = float(np.median(stats_stage.loc[mask, "token_count"].astype(float)))
            df_one["token_count_mean"] = float(np.mean(stats_stage.loc[mask, "token_count"].astype(float)))
            df_one["lexical_ttr_median"] = float(np.median(stats_stage.loc[mask, "lexical_ttr"].astype(float)))
            df_one["avg_token_len_median"] = float(np.median(stats_stage.loc[mask, "avg_token_len"].astype(float)))
            df_one["split_tag"] = SPLIT_TAG
            df_one["ood_name"] = ood_name
            df_one["backbone_name"] = config.model_name
            df_one["max_length"] = int(config.max_length)
            df_one["learning_rate"] = float(config.learning_rate)
            df_one["num_epochs"] = int(config.num_epochs)
            rows.append(df_one)

    return rows

primary_bin_rows = []
for model_label, meta in inference_cache.items():
    primary_bin_rows.extend(
        _build_bin_rows(model_label, df_test["prompt_text"].reset_index(drop=True), y_test, meta["test_score"], stage_name="TEST", ood_name="id")
    )
    primary_bin_rows.extend(
        _build_bin_rows(
            model_label,
            df_ood_map["ood_test"]["prompt_text"].reset_index(drop=True),
            y_ood_map["ood_test"],
            meta["ood_score_map"]["ood_test"],
            stage_name="OOD",
            ood_name="ood_test",
        )
    )

if primary_bin_rows:
    transformer_bins_df = pd.concat(primary_bin_rows, ignore_index=True)
    if not WRITE_MINIMAL_OUTPUTS:
        bins_path = OUT_DIR / f"metrics_transformer_bins_split{SPLIT_TAG}.csv"
        transformer_bins_df.to_csv(bins_path, index=False)
        print(f"Saved transformer bin metrics (primary): {bins_path}")
        bins_primary_suffix = OUT_DIR / f"metrics_transformer_bins_split{SPLIT_TAG}__ood-ood_test.csv"
        transformer_bins_df.to_csv(bins_primary_suffix, index=False)
        print(f"Saved transformer bin metrics primary-suffixed: {bins_primary_suffix}")
    else:
        print("WRITE_MINIMAL_OUTPUTS=True: skipped transformer bin metrics file outputs.")
    display(transformer_bins_df.head(12))
else:
    print("No transformer bin metrics were produced for primary OOD.")

for ood_name in [name for name in X_ood_text_map if name != "ood_test"]:
    ood_bin_rows = []
    for model_label, meta in inference_cache.items():
        ood_bin_rows.extend(
            _build_bin_rows(
                model_label,
                df_ood_map[ood_name]["prompt_text"].reset_index(drop=True),
                y_ood_map[ood_name],
                meta["ood_score_map"][ood_name],
                stage_name="OOD",
                ood_name=ood_name,
            )
        )
    if ood_bin_rows and (not WRITE_MINIMAL_OUTPUTS):
        df_extra_bins = pd.concat(ood_bin_rows, ignore_index=True)
        bins_extra_path = OUT_DIR / f"metrics_transformer_bins_split{SPLIT_TAG}__ood-{ood_name}.csv"
        df_extra_bins.to_csv(bins_extra_path, index=False)
        print(f"Saved transformer bin metrics ({ood_name}): {bins_extra_path}")


Saved transformer bin metrics (primary): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_bins_splitC.csv
Saved transformer bin metrics primary-suffixed: /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_bins_splitC__ood-ood_test.csv


,model,split,acc,macro_f1,auc_pr,roc_auc,tpr_at_1pct_fpr,tpr_at_5pct_fpr,tpr_at_10pct_fpr,semantic_coverage,...,token_count_median,token_count_mean,lexical_ttr_median,avg_token_len_median,split_tag,ood_name,backbone_name,max_length,learning_rate,num_epochs
0,TRANSFORMER_ROBERTA_BASE,TEST_BIN_LENGTH_BIN_LEN_Q1,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,8.5,7.368421,1.000000,5.111111,C,id,roberta-base,256,0.00002,3
1,TRANSFORMER_ROBERTA_BASE,TEST_BIN_LENGTH_BIN_LEN_Q2,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,11.0,10.918919,1.000000,5.100000,C,id,roberta-base,256,0.00002,3
2,TRANSFORMER_ROBERTA_BASE,TEST_BIN_LENGTH_BIN_LEN_Q3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,14.0,13.783784,0.928571,4.666667,C,id,roberta-base,256,0.00002,3
3,TRANSFORMER_ROBERTA_BASE,TEST_BIN_LENGTH_BIN_LEN_Q4,0.945946,0.931481,0.971276,0.944056,0.500000,0.500000,0.884615,None,...,17.0,25.027027,0.916667,4.750000,C,id,roberta-base,256,0.00002,3
4,TRANSFORMER_ROBERTA_BASE,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q1,0.973684,0.964912,0.998768,0.996429,0.964286,0.964286,1.000000,None,...,15.0,21.210526,0.888889,4.700000,C,id,roberta-base,256,0.00002,3
5,TRANSFORMER_ROBERTA_BASE,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q2,0.972973,0.953342,0.985683,0.947619,0.633333,0.633333,0.633333,None,...,14.0,14.864865,0.928571,4.823529,C,id,roberta-base,256,0.00002,3
6,TRANSFORMER_ROBERTA_BASE,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q3,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,11.0,11.594595,1.000000,5.100000,C,id,roberta-base,256,0.00002,3
7,TRANSFORMER_ROBERTA_BASE,TEST_BIN_COMPLEXITY_BIN_COMPLEX_Q4,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,None,...,9.0,9.054054,1.000000,5.375000,C,id,roberta-base,256,0.00002,3
8,TRANSFORMER_ROBERTA_BASE,OOD_BIN_LENGTH_BIN_LEN_Q1,0.822917,0.681250,0.579587,0.858449,0.178571,0.428571,0.500000,None,...,7.0,7.208333,1.000000,5.125000,C,ood_test,roberta-base,256,0.00002,3
9,TRANSFORMER_ROBERTA_BASE,OOD_BIN_LENGTH_BIN_LEN_Q2,0.718750,0.692600,0.828761,0.845899,0.380952,0.464286,0.488095,None,...,10.0,10.031250,1.000000,4.888889,C,ood_test,roberta-base,256,0.00002,3


Saved transformer bin metrics (ood_test_injection): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_bins_splitC__ood-ood_test_injection.csv


Saved transformer bin metrics (ood_test_injection_standard): /Users/timiakinrele/VSCode/dissertation/ai-jailbreak-classifier/experiments/results/metrics/metrics_transformer_bins_splitC__ood-ood_test_injection_standard.csv


<!-- NOTEBOOK_OUTPUT_SUMMARY -->
## 6. Output Summary
1. Writes canonical split-level transformer metrics to `experiments/results/metrics/metrics_transformer_split{tag}.csv`.
2. When `WRITE_MINIMAL_OUTPUTS=0`, also writes per-OOD suffixed transformer metrics and transformer difficulty-bin diagnostics.
3. Produces two supervised encoder baseline rows per split:
   - `TRANSFORMER_ROBERTA_BASE`
   - `TRANSFORMER_DEBERTA_BASE`
4. The resulting files are consumed by the repeatability and canonical-compaction steps alongside the existing ablation, semantic, and hybrid families.
